# Notebook 8 — Does Taylor importance change AFTER duplication?

**The question.** All our pruning has used Taylor importance computed on the dense, 36-layer model. But when we apply F duplication (layers 9-34 run twice, 62-layer stack), the model's forward dynamics change. Hidden states through later layers are different; gradient flows through duplicated layers TWICE (accumulating contributions from both passes).

If we compute Taylor on the duplicated model, the importance map could shift:
- **Duplicated layers** may look *more* important (gradient accumulates from two uses).
- **Non-duplicated layers** may see different hidden-state distributions, altering their relative importance.
- **Per-tile structure within layers** may reveal which sub-circuits matter most when the block runs twice.

**The action.** If the map is meaningfully different, using *post-duplication* Taylor scores to pick pruning masks could give better targets than the dense-only map — pruning the tiles that are genuinely expendable in the duplicated forward regime, rather than tiles that were expendable before duplication changed the dynamics.

## Structure

**Phase A — Importance map comparison.**
1. Compute Taylor on the clean 36-layer dense model → `importance_dense`.
2. Apply F duplication [9, 34]. Compute Taylor on the 62-layer duplicated model (weights still clean) → `importance_dup`.
3. Compare per-layer aggregates side by side. Visualize.

**Phase B — Pruning effectiveness comparison.**
For each sparsity X in {10, 20, 30}%:
- Prune with masks derived from `importance_dense`, F active → eval MMLU. (Already measured in notebook 7.)
- Prune with masks derived from `importance_dup`, F active → eval MMLU. *New.*
- Compare: does dup-aware pruning beat dense-aware pruning when F is on?

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, time
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

import config
from eval_mmlu import evaluate

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
TILE_R, TILE_C = config.TILE_SIZES[0]
F_RANGE = (9, 34)  # Notebook 6's real winning F
PRUNE_LEVELS = [0.10, 0.20, 0.30]
N_CAL = 512
SEQ_LEN_CAL = 128

torch.set_float32_matmul_precision("high")
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} / {total/1e9:.2f} GB")

## 1. Load model + cache ORIGINAL_MLP_WEIGHTS and ORIGINAL_LAYERS

In [ ]:
print(f"loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=getattr(torch, config.TORCH_DTYPE),
    device_map=config.DEVICE,
)
model.eval()

ORIGINAL_LAYERS = list(model.model.layers)
N_LAYERS = len(ORIGINAL_LAYERS)
ORIGINAL_LAYER_TYPES = (list(model.config.layer_types)
                       if getattr(model.config, "layer_types", None) is not None else None)

def is_mlp_name(name):
    return any(k in name for k in config.PRUNE_TARGETS_PATTERNS)

ORIGINAL_MLP_WEIGHTS = {}
for name, module in model.named_modules():
    if isinstance(module, nn.Linear) and is_mlp_name(name):
        ORIGINAL_MLP_WEIGHTS[name + ".weight"] = module.weight.data.detach().clone().cpu()

print(f"model: {N_LAYERS} layers, {sum(p.numel() for p in model.parameters())/1e9:.2f}B params")
print(f"cached {len(ORIGINAL_MLP_WEIGHTS)} MLP matrices on CPU")
print(f"peak GPU: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

## 2. Calibration data (same C4 stream as notebooks 5/6/7)

In [ ]:
cal_ds = load_dataset(config.CALIBRATION_DATASET, config.CALIBRATION_SUBSET,
                     split="train", streaming=True, trust_remote_code=True)
cal_texts = []
for ex in cal_ds:
    t = ex.get("text", "")
    if len(t) > 50:
        cal_texts.append(t)
    if len(cal_texts) >= N_CAL:
        break
cal_encodings = [tokenizer(t, return_tensors="pt", max_length=SEQ_LEN_CAL, truncation=True)
                 for t in cal_texts]
print(f"calibration samples: {len(cal_encodings)}")

## 3. Utilities — reusable Taylor calibration + duplication + pruning

`compute_taylor()` runs the full forward+backward Taylor pass on whatever current model state is (dense or duplicated). Returns per-matrix tile scores on CPU.

`set_F()` / `reset_layers()` — standard duplication helpers, sync `layer_types`.

`restore_mlp_weights()` — copies cached clean weights back to GPU. Required between pruning tests.

`apply_taylor_masks(importance_norm, sparsity)` — zeros the bottom-sparsity fraction of each MLP matrix.

In [ ]:
def get_component_type(name):
    for p in config.PRUNE_TARGETS_PATTERNS:
        if p in name:
            return p
    return "unknown"

def get_layer_idx(name):
    parts = name.split(".")
    i = parts.index("layers")
    return int(parts[i+1])

# ── duplication ───────────────────────────────────────────────────────

def set_duplication_range(start, end):
    new_layers = (list(ORIGINAL_LAYERS[:end+1])
                  + list(ORIGINAL_LAYERS[start:end+1])
                  + list(ORIGINAL_LAYERS[end+1:]))
    model.model.layers = nn.ModuleList(new_layers)
    model.config.num_hidden_layers = len(new_layers)
    if ORIGINAL_LAYER_TYPES is not None:
        model.config.layer_types = (ORIGINAL_LAYER_TYPES[:end+1]
                                    + ORIGINAL_LAYER_TYPES[start:end+1]
                                    + ORIGINAL_LAYER_TYPES[end+1:])

def reset_layers():
    model.model.layers = nn.ModuleList(ORIGINAL_LAYERS)
    model.config.num_hidden_layers = len(ORIGINAL_LAYERS)
    if ORIGINAL_LAYER_TYPES is not None:
        model.config.layer_types = list(ORIGINAL_LAYER_TYPES)

def set_F():
    set_duplication_range(*F_RANGE)

# ── pruning / restore ─────────────────────────────────────────────────

def normalize_importance(importance_raw):
    """Per-component-type z-normalization — same scheme as all prior notebooks."""
    comp_stats = {}
    for ct in config.PRUNE_TARGETS_PATTERNS:
        vals = np.concatenate([m.ravel() for n, m in importance_raw.items() if ct in n])
        comp_stats[ct] = (vals.mean(), vals.std())
    norm = {}
    for name, m in importance_raw.items():
        mu, sd = comp_stats[get_component_type(name)]
        norm[name] = (m - mu) / sd if sd > 1e-8 else np.zeros_like(m)
    return norm

def per_layer_scores(importance_norm):
    ls = np.zeros(N_LAYERS)
    for name, m in importance_norm.items():
        ls[get_layer_idx(name)] += m.sum()
    return ls

def build_masks(importance_norm, sparsity):
    masks = {}
    for name, m in importance_norm.items():
        thr = float(np.percentile(m.ravel(), sparsity * 100))
        masks[name] = m < thr
    return masks

def apply_masks(masks):
    with torch.no_grad():
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                pn = name + ".weight"
                if pn in masks:
                    mask = masks[pn]
                    w = module.weight.data
                    out_f, in_f = w.shape
                    nr, nc = out_f // TILE_R, in_f // TILE_C
                    for i in range(nr):
                        for j in range(nc):
                            if mask[i, j]:
                                w[i*TILE_R:(i+1)*TILE_R, j*TILE_C:(j+1)*TILE_C] = 0

def restore_mlp_weights():
    with torch.no_grad():
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                pn = name + ".weight"
                if pn in ORIGINAL_MLP_WEIGHTS:
                    module.weight.data.copy_(ORIGINAL_MLP_WEIGHTS[pn].to(module.weight.device))

# ── Taylor calibration (reusable) ─────────────────────────────────────

def compute_taylor(model, cal_encodings, tag="taylor"):
    """Run Taylor calibration on current model state. Returns {name: tile_scores (np.ndarray)}.
    Works identically on dense (36-layer) and duplicated (62-layer) models — duplicated
    modules share parameter refs, so gradients accumulate naturally on the unique weights."""
    for p in model.parameters():
        p.requires_grad_(False)
    for name, p in model.named_parameters():
        if is_mlp_name(name) and p.ndim == 2:
            p.requires_grad_(True)

    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    importance_gpu = {}
    def make_hook(pname):
        def hook(param):
            if param.grad is None:
                return
            taylor = (param.data * param.grad).abs().float()
            out_dim, in_dim = taylor.shape
            nr, nc = out_dim // TILE_R, in_dim // TILE_C
            tt = taylor[:nr*TILE_R, :nc*TILE_C].reshape(nr, TILE_R, nc, TILE_C).permute(0, 2, 1, 3)
            tile_scores = tt.reshape(nr, nc, -1).sum(dim=-1)
            if pname in importance_gpu:
                importance_gpu[pname] += tile_scores
            else:
                importance_gpu[pname] = tile_scores.clone()
            param.grad = None
        return hook

    hook_handles = []
    for name, p in model.named_parameters():
        if is_mlp_name(name) and p.ndim == 2 and p.requires_grad:
            hook_handles.append(p.register_post_accumulate_grad_hook(make_hook(name)))

    n_samples = 0
    for enc in tqdm(cal_encodings, desc=tag):
        model.zero_grad(set_to_none=True)
        inputs = {k: v.to(config.DEVICE) for k, v in enc.items()}
        out = model(**inputs, labels=inputs["input_ids"])
        out.loss.backward()
        n_samples += 1

    for h in hook_handles:
        h.remove()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    for p in model.parameters():
        p.requires_grad_(False)

    importance = {name: (v / n_samples).cpu().numpy() for name, v in importance_gpu.items()}
    del importance_gpu
    gc.collect(); torch.cuda.empty_cache()
    return importance

def eval_tag(tag):
    t0 = time.time()
    results = evaluate(model, tokenizer, tag=tag)
    acc = results["overall"]["accuracy"]
    print(f"  {tag:<36s} MMLU {acc*100:5.2f}%   ({time.time()-t0:.1f}s)")
    return acc

## 4. Taylor on the DENSE 36-layer model

This is the same calibration as notebooks 5/6/7. Serves as the baseline importance map.

In [ ]:
reset_layers()
restore_mlp_weights()

print("=== Taylor #1 — DENSE 36-layer model ===")
importance_dense_raw = compute_taylor(model, cal_encodings, tag="taylor_dense")
importance_dense = normalize_importance(importance_dense_raw)
layer_scores_dense = per_layer_scores(importance_dense)
print(f"peak GPU: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

## 5. Taylor on the DUPLICATED 62-layer model (F applied)

Weights are still clean (no pruning yet). The model now runs layers 9-34 twice. Gradients accumulate on the shared parameters from both forward passes.

In [ ]:
set_F()
print(f"applied F: {len(model.model.layers)} layers active (expect 62)")

print("\n=== Taylor #2 — DUPLICATED 62-layer model ===")
importance_dup_raw = compute_taylor(model, cal_encodings, tag="taylor_dup")
importance_dup = normalize_importance(importance_dup_raw)
layer_scores_dup = per_layer_scores(importance_dup)
print(f"peak GPU: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

## 6. Compare per-layer scores

In [ ]:
print("layer | dense | dup   | Δ      | in_F?")
print("------+-------+-------+--------+------")
for i in range(N_LAYERS):
    d = layer_scores_dense[i]
    u = layer_scores_dup[i]
    in_f = F_RANGE[0] <= i <= F_RANGE[1]
    marker = "★" if in_f else " "
    print(f" {i:2d}   | {d:+6.0f} | {u:+6.0f} | {u-d:+6.0f} | {marker}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

in_f_mask = np.array([F_RANGE[0] <= i <= F_RANGE[1] for i in range(N_LAYERS)])
colors_dense = ["#1f77b4" if in_f_mask[i] else "#999" for i in range(N_LAYERS)]
colors_dup   = ["#d62728" if in_f_mask[i] else "#999" for i in range(N_LAYERS)]

axes[0].bar(range(N_LAYERS), layer_scores_dense, color=colors_dense)
axes[0].axhline(0, color="k", lw=0.5)
axes[0].set_ylabel("Σ z-norm Taylor")
axes[0].set_title(f"Taylor importance on DENSE (36 layers). Blue bars = F range [{F_RANGE[0]}, {F_RANGE[1]}]")

axes[1].bar(range(N_LAYERS), layer_scores_dup, color=colors_dup)
axes[1].axhline(0, color="k", lw=0.5)
axes[1].set_xlabel("layer index")
axes[1].set_ylabel("Σ z-norm Taylor")
axes[1].set_title(f"Taylor importance on DUPLICATED (62 layers, F applied). Red bars = F range")

plt.tight_layout(); plt.show()

# Per-layer delta
fig2, ax = plt.subplots(figsize=(11, 2.5))
delta = layer_scores_dup - layer_scores_dense
colors_d = ["#d62728" if in_f_mask[i] else "#1f77b4" for i in range(N_LAYERS)]
ax.bar(range(N_LAYERS), delta, color=colors_d)
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("layer index")
ax.set_ylabel("dup − dense")
ax.set_title("Per-layer Δ Taylor importance (red = in F duplication range)")
plt.tight_layout(); plt.show()

## 7. Phase B — Pruning comparison: dense-Taylor vs dup-Taylor masks (F active)

For each sparsity in {10, 20, 30}%:
- Build masks from `importance_dense` → apply → F active → eval MMLU.
- Build masks from `importance_dup` → apply → F active → eval MMLU.

The dense-Taylor masks should reproduce notebook 7's numbers (Taylor X% + F): 34.57% / 25.54% / 24.01% at 10/20/30%.
The dup-Taylor masks are new — tests whether dup-aware scoring picks better tiles to prune.

In [ ]:
# Evaluate sanity baselines first: dense (no F, no prune), F only
reset_layers()
restore_mlp_weights()
results = {}
results["dense"] = eval_tag("pB_dense")

set_F()
results["F_only"] = eval_tag("pB_F_only")  # expect 48.87%

# Pruning sweep — keep F applied throughout
for sp in PRUNE_LEVELS:
    # ── dense-Taylor masks
    restore_mlp_weights()
    apply_masks(build_masks(importance_dense, sp))
    results[f"T{int(sp*100)}_dense_masks_F"] = eval_tag(f"pB_T{int(sp*100)}_denseMask_F")

    # ── dup-Taylor masks
    restore_mlp_weights()
    apply_masks(build_masks(importance_dup, sp))
    results[f"T{int(sp*100)}_dup_masks_F"] = eval_tag(f"pB_T{int(sp*100)}_dupMask_F")

# Cleanup
restore_mlp_weights()
reset_layers()
print("\nPhase B done.")

## 8. Summary

In [ ]:
print("=" * 72)
print(f"  Taylor AFTER duplication — does the importance map change meaningfully?")
print("=" * 72)
print(f"  Dense baseline                         {results['dense']*100:6.2f}%")
print(f"  F only (layers 9-34 dup)               {results['F_only']*100:6.2f}%  (Δ {(results['F_only']-results['dense'])*100:+.2f}pp)")
print()
print(f"  {'Sparsity':<10s}  {'dense-Taylor masks':<22s}  {'dup-Taylor masks':<22s}  {'dup − dense':>12s}")
print("-" * 72)
for sp in PRUNE_LEVELS:
    d = results[f"T{int(sp*100)}_dense_masks_F"]
    u = results[f"T{int(sp*100)}_dup_masks_F"]
    diff = (u - d) * 100
    print(f"  {int(sp*100):>3d}%       {d*100:>6.2f}% + F            {u*100:>6.2f}% + F            {diff:+6.2f}pp")
print("=" * 72)

# Per-layer change correlation
in_f = np.array([F_RANGE[0] <= i <= F_RANGE[1] for i in range(N_LAYERS)])
delta_in_f = (layer_scores_dup[in_f] - layer_scores_dense[in_f])
delta_out_f = (layer_scores_dup[~in_f] - layer_scores_dense[~in_f])
print(f"\n  Per-layer Δ (dup − dense):")
print(f"    inside F range [{F_RANGE[0]},{F_RANGE[1]}]: mean Δ = {delta_in_f.mean():+.1f}  (n={in_f.sum()})")
print(f"    outside F range:              mean Δ = {delta_out_f.mean():+.1f}  (n={(~in_f).sum()})")